# Modeling Experiments.

This notebook summarizes the results generated by `src/models/run_experiments.py`. We will start from the saved artifacts: metrics, summary, predictions, and models to:

1. Visualize and compare metrics by model/fold.
2. Analyze the errors on the test set (residuals, and y_true vs y_pred scatter).
3. Review feature importance (if available for the model) and document conclusions.

### 1. Setup
Update the `EXPERIMENT_DIR` path with the specific folder (e.g., `experiment_YYYYMMDD_HHMMSS`).

In [28]:
import json
from pathlib import Path

import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

EXPERIMENT_BASE_DIR = Path("../data/results/modeling/experiments")
runner_experiments = sorted(
    [d for d in EXPERIMENT_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("runner_id_")],
    reverse=True,
)
if not runner_experiments:
    raise FileNotFoundError("No runner_id experiments found. Run run_experiments.py first.")

SELECTORS = [
    {"key": "fatigue_full", "target": "fatigue_score", "label": "Fatigue Score", "dataset_suffix": "features_dataset.parquet"},
    {"key": "fatigue_no_orient", "target": "fatigue_score", "label": "Fatigue Score (sin orientación)", "dataset_suffix": "features_dataset_3s_50olap_without_orientation.parquet"},
    {"key": "reported", "target": "reported_rpe", "label": "RPE", "dataset_suffix": None},
]

experiment_data = {}
for sel in SELECTORS:
    target_lower = sel["target"].lower()
    dataset_suffix = sel["dataset_suffix"]
    selected_dir = None
    for exp_dir in runner_experiments:
        cfg_path = exp_dir / "config.json"
        if not cfg_path.exists():
            continue
        cfg = json.loads(cfg_path.read_text())
        dataset_path = str(cfg.get("dataset", ""))
        if cfg.get("target", "reported_rpe").lower() == target_lower:
            if dataset_suffix is None or dataset_suffix in dataset_path:
                selected_dir = exp_dir
                break
    if not selected_dir:
        print(f"No experiment found for {sel['label']}.")
        continue

    metrics_path = selected_dir / "metrics.csv"
    summary_path = selected_dir / "summary.csv"
    predictions_path = selected_dir / "predictions.parquet"
    feature_cols_path = selected_dir / "feature_columns.json"

    experiment_data[sel["key"]] = {
        "dir": selected_dir,
        "metrics": pd.read_csv(metrics_path),
        "summary": pd.read_csv(summary_path),
        "pred": pd.read_parquet(predictions_path),
        "feature_columns": json.loads(feature_cols_path.read_text()) if feature_cols_path.exists() else None,
        "target_label": sel["label"],
    }
    print(f"Using experiment '{selected_dir.name}' for {sel['label']}")

if not experiment_data:
    raise FileNotFoundError("No experiments found for the requested selectors.")


Using experiment 'runner_id_20251119_193024' for Fatigue Score
No experiment found for Fatigue Score (sin orientación).
No experiment found for RPE.


### 2. Metrics Comparison
Charts to compare MAE/RMSE/R² by model and split.

In [29]:
for target, data in experiment_data.items():
    summary_df = data["summary"].copy()
    label = data["target_label"]
    print(f"Target: {label} (Experiment: {data['dir'].name})")

    fig_mae = px.bar(
        summary_df,
        x="model",
        y="mae_mean",
        color="split",
        error_y="mae_std",
        title=f"MAE by model and split ({label})",
    )
    fig_mae.show()

    fig_rmse = px.bar(
        summary_df,
        x="model",
        y="rmse_mean",
        color="split",
        error_y="rmse_std",
        title=f"RMSE by model and split ({label})",
    )
    fig_rmse.show()

    fig_r2 = px.bar(
        summary_df,
        x="model",
        y="r2_mean",
        color="split",
        error_y="r2_std",
        title=f"R² by model and split ({label})",
    )
    fig_r2.show()


Target: Fatigue Score (Experiment: runner_id_20251119_193024)


### 3. Residuals and scatters (test)

We inspect how each model performs on the test set.

In [30]:
for target, data in experiment_data.items():
    pred_df = data["pred"].copy()
    label = data["target_label"]
    pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

    fig = px.scatter(
        pred_df,
        x="y_true",
        y="y_pred",
        color="model",
        title=f"True vs Predicted ({label}, test)",
        labels={"y_true": f"{label} (true)", "y_pred": f"{label} (predicted)"},
    )
    fig.add_trace(
        go.Scatter(
            x=[pred_df.y_true.min(), pred_df.y_true.max()],
            y=[pred_df.y_true.min(), pred_df.y_true.max()],
            mode="lines",
            name="Ideal",
        )
    )
    fig.show()

    fig_res = px.box(
        pred_df,
        x="model",
        y="residual",
        title=f"Residual distribution ({label}, test)",
    )
    fig_res.show()


### 4. Feature Importance




In [31]:
import joblib
import numpy as np

for target, data in experiment_data.items():
    label = data["target_label"]
    feature_columns = data["feature_columns"]
    available_models = sorted(data["summary"]["model"].unique())
    print(f"Target: {label} (Experiment: {data['dir'].name})")

    for model_name in available_models:
        model_path = data["dir"] / f"{model_name}_best.joblib"
        if not model_path.exists():
            print(f"  Model artifact not found for {model_name}.")
            continue

        pipeline = joblib.load(model_path)
        model = pipeline.named_steps["model"]
        importances = getattr(model, "feature_importances_", None)
        if importances is None:
            coef = getattr(model, "coef_", None)
            if coef is not None:
                importances = np.abs(np.ravel(coef))
            else:
                print(f"  {model_name} does not expose feature importances or coefficients.")
                continue

        feature_names = (
            feature_columns
            if feature_columns is not None and len(feature_columns) == len(importances)
            else [f"f{i}" for i in range(len(importances))]
        )

        fi = (
            pd.DataFrame({"feature": feature_names, "importance": importances})
            .sort_values("importance", ascending=False)
            .head(20)
        )

        px.bar(fi, x="feature", y="importance", title=f"Top features ({model_name}, {label})").show()


Target: Fatigue Score (Experiment: runner_id_20251119_193024)


  hist_gradient_boosting does not expose feature importances or coefficients.
